# 04 - 勝率推移・トレンド分析

時間軸に沿った勝率の変化、時間帯別パフォーマンス、連勝/連敗ストリークを分析する。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import yaml
from pathlib import Path

for font in ['Meiryo', 'Yu Gothic', 'MS Gothic', 'Hiragino Sans']:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        plt.rcParams['font.family'] = font
        break
    except ValueError:
        continue
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])

DATA = Path('..') / 'data' / 'processed'
CONFIG = Path('..') / 'config' / 'settings.yaml'

with open(CONFIG, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
MEMBERS = {f'{m["game_name"]}#{m["tag_line"]}' for m in cfg['members']}

df = pd.read_csv(DATA / 'player_stats.csv', parse_dates=['gameCreation'])
df['riotId'] = df['summonerName'] + '#' + df['tagLine']
df_m = df[df['riotId'].isin(MEMBERS)].copy()
df_m.sort_values('gameCreation', inplace=True)

In [ ]:
# 週別勝率推移
df_m['week'] = df_m['gameCreation'].dt.isocalendar().week.astype(int)
df_m['year_week'] = df_m['gameCreation'].dt.strftime('%Y-W%U')

weekly = df_m.groupby('year_week').agg(
    games=('win', 'count'),
    winrate=('win', 'mean'),
).reset_index()
weekly['winrate_pct'] = (weekly['winrate'] * 100).round(1)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(weekly['year_week'], weekly['games'], alpha=0.3, label='試合数')
ax2.plot(weekly['year_week'], weekly['winrate_pct'], 'o-', color='red', label='勝率')
ax2.axhline(y=50, color='gray', linestyle='--', alpha=0.5)

ax1.set_xlabel('週')
ax1.set_ylabel('試合数')
ax2.set_ylabel('勝率 %')
ax1.set_title('週別 試合数と勝率の推移')
ax1.tick_params(axis='x', rotation=45)
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.show()

In [ ]:
# 時間帯 × 曜日 勝率ヒートマップ
df_m['hour'] = df_m['gameCreation'].dt.hour
df_m['dayofweek'] = df_m['gameCreation'].dt.day_name()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
heatmap_data = df_m.groupby(['dayofweek', 'hour'])['win'].mean().unstack(fill_value=0.5)
heatmap_data = heatmap_data.reindex(day_order) * 100

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(heatmap_data, cmap='RdYlGn', center=50, annot=True, fmt='.0f',
            linewidths=0.5, ax=ax)
ax.set_title('曜日 × 時間帯 勝率ヒートマップ (%)')
ax.set_xlabel('時間帯')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# 連勝 / 連敗ストリーク分析
unique_matches = df_m.drop_duplicates('matchId').sort_values('gameCreation')
wins = unique_matches['win'].astype(int).values

streaks = []
current_streak = 1
for i in range(1, len(wins)):
    if wins[i] == wins[i-1]:
        current_streak += 1
    else:
        streaks.append(('勝ち' if wins[i-1] else '負け', current_streak))
        current_streak = 1
streaks.append(('勝ち' if wins[-1] else '負け', current_streak))

df_streaks = pd.DataFrame(streaks, columns=['タイプ', '連続数'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, stype, color in zip(axes, ['勝ち', '負け'], ['#2ecc71', '#e74c3c']):
    data = df_streaks[df_streaks['タイプ'] == stype]['連続数']
    ax.hist(data, bins=range(1, data.max() + 2), color=color, alpha=0.7, edgecolor='white')
    ax.set_title(f'連{stype}ストリーク分布')
    ax.set_xlabel('連続数')
    ax.set_ylabel('回数')

plt.tight_layout()
plt.show()

print(f"最長連勝: {df_streaks[df_streaks['タイプ']=='勝ち']['連続数'].max()}")
print(f"最長連敗: {df_streaks[df_streaks['タイプ']=='負け']['連続数'].max()}")